In [ ]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from math import ceil, sqrt
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

from scipy.stats import mannwhitneyu


def compare_unmasked_counts_unpaired(
    df_A, df_B,
    pid_col='pid', item_col='varis', mask_col='mask',
    min_samples=5,
    fdr_method='fdr_bh'
):
    # unmasked only
    A = df_A.loc[~df_A[mask_col], [pid_col, item_col]]
    B = df_B.loc[~df_B[mask_col], [pid_col, item_col]]

    # pid × varis counts
    gA = A.groupby([pid_col, item_col]).size().reset_index(name='count_A')
    gB = B.groupby([pid_col, item_col]).size().reset_index(name='count_B')

    results = []

    all_items = sorted(set(gA[item_col]).union(set(gB[item_col])))

    for varis in all_items:
        a = gA[gA[item_col] == varis]['count_A'].values
        b = gB[gB[item_col] == varis]['count_B'].values

        if len(a) < min_samples or len(b) < min_samples:
            p = np.nan
        else:
            try:
                _, p = mannwhitneyu(a, b, alternative='two-sided')
            except ValueError:
                p = np.nan

        results.append({
            item_col: varis,
            'mean_A': a.mean() if len(a) > 0 else 0.0,
            'mean_B': b.mean() if len(b) > 0 else 0.0,
            'median_A': np.median(a) if len(a) > 0 else 0.0,
            'median_B': np.median(b) if len(b) > 0 else 0.0,
            'n_pid_A': len(a),
            'n_pid_B': len(b),
            'p_value': p
        })

    res = pd.DataFrame(results)

    # FDR
    res['p_fdr'] = np.nan
    valid = res['p_value'].notna()
    if valid.sum() > 0:
        res.loc[valid, 'p_fdr'] = multipletests(
            res.loc[valid, 'p_value'], method=fdr_method
        )[1]

    return res.sort_values(['p_fdr', 'p_value'], na_position='last').reset_index(drop=True)


def masked_reconstruction_mse(
    df,
    var_names=None,
    pid_filter=None,        # int or list-like: 계산 대상 pid 제한
    pid_target=None,        # 시각화할 특정 pid (None이면 시각화 안 함)
    plot_only_vars=None,    # 시각화할 varis list (None이면 pid 내 전체 var)
    time_col='times',
    var_col='varis',
    value_col='values',
    mask_col='mask',
    time_unit="hours",
    figsize_per_plot=(4, 3),
    sharey=False,
):
    """
    Unmasked 관측치를 knot로 하는 piecewise-linear(선형보간)로 원본을 복원했을 때,
    masked points에서만 MSE를 계산 (복원 오차).

    - unmasked knot 0개: 해당 (pid,var)에서는 복원 불가 -> MSE=NaN
    - unmasked knot 1개: 수평선 (그 한 점 값)으로 복원
    - unmasked knot >=2: 선형보간
    - MSE는 masked points에서만 계산

    Returns
    -------
    var_mse_df : DataFrame
        columns: [varis, var_name, n_masked_points, n_pid_valid, mse_masked_mean, mse_masked_std]
        (pid별 masked MSE를 만든 뒤 varis별로 평균/표준편차)
    overall : dict
        {
          'overall_mse_masked': float,
          'overall_n_masked_points': int,
          'overall_n_pid_var_valid': int
        }
    pid_var_detail : DataFrame
        (pid,varis)별 masked MSE 상세
        columns: [pid, varis, mse_masked, n_masked, n_unmasked, n_all]
    """

    required = {'pid', time_col, var_col, value_col, mask_col}
    if not required.issubset(df.columns):
        raise ValueError(f"df must contain columns {required}")

    d = df.copy()

    # pid filter for computation
    if pid_filter is not None:
        if np.isscalar(pid_filter):
            d = d[d['pid'] == pid_filter]
        else:
            d = d[d['pid'].isin(pid_filter)]

    if d.empty:
        raise ValueError("Filtered df is empty.")

    # 정렬
    d = d.sort_values(['pid', var_col, time_col]).reset_index(drop=True)

    # ---------- (1) 계산 파트: (pid,var)별 masked-only MSE ----------
    rows = []
    # overall용: masked 점들의 (y - yhat)^2 누적
    sse_total = 0.0
    n_masked_total = 0
    n_pid_var_valid = 0  # masked가 존재하고, 복원이 가능한 pid-var count

    for (pid, varid), g in d.groupby(['pid', var_col], sort=False):
        g = g.sort_values(time_col)
        t_all = g[time_col].to_numpy()
        y_all = g[value_col].to_numpy()

        g_un = g[~g[mask_col]]           # knots
        g_mk = g[g[mask_col]]            # masked points (target of error)

        n_all = len(g)
        n_un = len(g_un)
        n_mk = len(g_mk)

        # masked 포인트가 없으면 "복원 오차" 개념 자체가 없음
        if n_mk == 0:
            mse_mk = np.nan
            rows.append({
                'pid': pid, var_col: varid,
                'mse_masked': mse_mk,
                'n_masked': n_mk, 'n_unmasked': n_un, 'n_all': n_all
            })
            continue

        # 복원 불가: knot 0개
        if n_un == 0:
            mse_mk = np.nan
            rows.append({
                'pid': pid, var_col: varid,
                'mse_masked': mse_mk,
                'n_masked': n_mk, 'n_unmasked': n_un, 'n_all': n_all
            })
            continue

        # knot 1개면 수평선
        if n_un == 1:
            y_hat_mk = np.full(shape=n_mk, fill_value=g_un[value_col].iloc[0], dtype=float)
        else:
            # knot time 중복이면 평균으로 축약
            t_k = g_un[time_col].to_numpy()
            y_k = g_un[value_col].to_numpy()

            if len(np.unique(t_k)) != len(t_k):
                tmp = (
                    pd.DataFrame({time_col: t_k, 'yk': y_k})
                    .groupby(time_col)['yk'].mean()
                    .reset_index()
                )
                t_k = tmp[time_col].to_numpy()
                y_k = tmp['yk'].to_numpy()

            order = np.argsort(t_k)
            t_k = t_k[order]
            y_k = y_k[order]

            # masked time에서의 예측값(범위 밖은 양끝값)
            t_mk = g_mk[time_col].to_numpy()
            y_hat_mk = np.interp(t_mk, t_k, y_k)

        y_mk = g_mk[value_col].to_numpy()
        se = (y_mk - y_hat_mk) ** 2
        mse_mk = float(np.mean(se))

        # overall 누적
        sse_total += float(np.sum(se))
        n_masked_total += int(n_mk)
        n_pid_var_valid += 1

        rows.append({
            'pid': pid, var_col: varid,
            'mse_masked': mse_mk,
            'n_masked': n_mk, 'n_unmasked': n_un, 'n_all': n_all
        })

    pid_var_detail = pd.DataFrame(rows)

    # ---------- (2) varis별 요약 ----------
    # mse_masked는 pid별 값이므로 varis별로 평균/표준편차
    agg = (
        pid_var_detail
        .groupby(var_col)
        .agg(
            n_pid_valid=('mse_masked', lambda s: int(s.notna().sum())),
            n_masked_points=('n_masked', 'sum'),
            mse_masked_mean=('mse_masked', 'mean'),
            mse_masked_std=('mse_masked', 'std'),
        )
        .reset_index()
    )

    if var_names:
        agg['var_name'] = agg[var_col].map(lambda x: var_names.get(x, f"var {x}"))
    else:
        agg['var_name'] = agg[var_col].map(lambda x: f"var {x}")

    var_mse_df = agg[[var_col, 'var_name', 'n_masked_points', 'n_pid_valid', 'mse_masked_mean', 'mse_masked_std']] \
        .sort_values('mse_masked_mean', ascending=False) \
        .reset_index(drop=True)

    # ---------- (3) 전체 MSE (masked-only) ----------
    overall = {
        'overall_mse_masked': (sse_total / n_masked_total) if n_masked_total > 0 else np.nan,
        'overall_n_masked_points': int(n_masked_total),
        'overall_n_pid_var_valid': int(n_pid_var_valid),
    }

    # ---------- (4) 선택적 pid 시각화 ----------
    if pid_target is not None:
        df_pid = d[d['pid'] == pid_target].copy()
        if df_pid.empty:
            raise ValueError(f"pid={pid_target} 에 해당하는 데이터가 없습니다.")

        df_pid = df_pid.sort_values([var_col, time_col])

        uniq_vars = df_pid[var_col].unique()
        if plot_only_vars is not None:
            plot_set = set(plot_only_vars)
            uniq_vars = [v for v in uniq_vars if v in plot_set]

        nplots = len(uniq_vars)
        if nplots == 0:
            raise ValueError("No variables to plot after filtering (plot_only_vars).")

        cols = int(ceil(sqrt(nplots)))
        rows_ = int(ceil(nplots / cols))

        fig, axes = plt.subplots(
            rows_, cols,
            figsize=(figsize_per_plot[0]*cols, figsize_per_plot[1]*rows_),
            sharey=sharey
        )
        if not isinstance(axes, np.ndarray):
            axes = np.array([axes])
        axes = axes.reshape(rows_, cols)

        handles, labels = [], []

        for idx, var_id in enumerate(uniq_vars):
            r, c = divmod(idx, cols)
            ax = axes[r, c]

            g = df_pid[df_pid[var_col] == var_id].sort_values(time_col)
            t_all = g[time_col].to_numpy()
            y_all = g[value_col].to_numpy()

            g_un = g[~g[mask_col]]
            g_mk = g[g[mask_col]]

            # all obs (faint)
            ax.scatter(t_all, y_all, marker='.', alpha=0.25, label='all obs')

            # knots + masked
            if not g_un.empty:
                ax.scatter(g_un[time_col], g_un[value_col], marker='o', alpha=0.9, label='unmasked(knot)')
            if not g_mk.empty:
                ax.scatter(g_mk[time_col], g_mk[value_col], marker='x', alpha=0.9, label='masked(target)')

            # reconstruction line
            if len(g_un) >= 1:
                if len(g_un) == 1:
                    y_hat_all = np.full_like(y_all, fill_value=g_un[value_col].iloc[0], dtype=float)
                else:
                    t_k = g_un[time_col].to_numpy()
                    y_k = g_un[value_col].to_numpy()

                    if len(np.unique(t_k)) != len(t_k):
                        tmp = (
                            pd.DataFrame({time_col: t_k, 'yk': y_k})
                            .groupby(time_col)['yk'].mean()
                            .reset_index()
                        )
                        t_k = tmp[time_col].to_numpy()
                        y_k = tmp['yk'].to_numpy()

                    order = np.argsort(t_k)
                    t_k = t_k[order]
                    y_k = y_k[order]

                    y_hat_all = np.interp(t_all, t_k, y_k)

                ax.plot(t_all, y_hat_all, linewidth=1.2, label='recon (PWL)')

            name = var_names.get(var_id, f"var {var_id}") if var_names else f"var {var_id}"
            ax.set_title(name)
            ax.set_xlabel(f"time ({time_unit})")
            ax.set_ylabel("value")
            ax.set_xlim(0.0, 1.0)

            ax.grid(True, linestyle='--', linewidth=0.5, alpha=0.5)

            for h, l in zip(*ax.get_legend_handles_labels()):
                if l not in labels:
                    handles.append(h); labels.append(l)

        for j in range(nplots, rows_*cols):
            r, c = divmod(j, cols)
            axes[r, c].axis('off')

        fig.legend(handles, labels, loc='lower right', ncol=2, frameon=False, fontsize=10)
        fig.suptitle(f"pid={pid_target} masked-only reconstruction check", y=0.95, fontsize=12)
        fig.tight_layout(rect=[0, 0, 1, 0.95])
        plt.show()

    return var_mse_df, overall, pid_var_detail

def compare_unmasked_counts(
    df_A, df_B,
    pid_col='pid', item_col='varis', mask_col='mask',
    join_how='outer',      # 'outer' 추천(한쪽은 0개인 경우도 비교)
    min_pairs=5,
    fdr_method='fdr_bh'
):
    """
    각 varis별로 PID 단위 unmasked 관측치 개수 분포를 비교 (A vs B, paired).
    - unmasked := mask == False
    - item별 Wilcoxon + FDR
    """

    # 0) unmasked만 남김
    A = df_A.loc[~df_A[mask_col], [pid_col, item_col]]
    B = df_B.loc[~df_B[mask_col], [pid_col, item_col]]

    # 1) PID×varis unmasked count
    gA = A.groupby([pid_col, item_col]).size().reset_index(name='count_A')
    gB = B.groupby([pid_col, item_col]).size().reset_index(name='count_B')

    # 2) align pairs
    merged = gA.merge(gB, on=[pid_col, item_col], how=join_how)
    if join_how == 'outer':
        merged['count_A'] = merged['count_A'].fillna(0).astype(int)
        merged['count_B'] = merged['count_B'].fillna(0).astype(int)

    if merged.empty:
        return pd.DataFrame(columns=[
            item_col, 'n_pids', 'mean_A', 'mean_B', 'mean_diff', 'p_value', 'p_fdr'
        ])

    # 3) item별 paired test
    rows = []
    for varis, grp in merged.groupby(item_col, sort=False):
        a = grp['count_A'].to_numpy()
        b = grp['count_B'].to_numpy()
        n = len(a)

        mean_A = a.mean()
        mean_B = b.mean()
        mean_diff = (a - b).mean()

        if n < min_pairs:
            p = np.nan
        else:
            if np.all(a == b):
                p = 1.0
            else:
                try:
                    _, p = wilcoxon(a, b, zero_method='pratt')
                except ValueError:
                    p = np.nan

        rows.append([varis, n, mean_A, mean_B, mean_diff, p])

    res = pd.DataFrame(rows, columns=[
        item_col, 'n_pids', 'mean_A', 'mean_B', 'mean_diff', 'p_value'
    ])

    # 4) FDR (p-value 있는 것만)
    res['p_fdr'] = np.nan
    valid = res['p_value'].notna()
    if valid.sum() > 0:
        res.loc[valid, 'p_fdr'] = multipletests(
            res.loc[valid, 'p_value'].values,
            method=fdr_method
        )[1]

    return res.sort_values(['p_fdr', 'p_value'], na_position='last').reset_index(drop=True)

def plot_patient_by_var_df(df, pid_target, graph_name=None, var_names=None, time_unit="hours",
                           figsize_per_plot=(4, 3), sharey=False):
    """
    DataFrame 기반 시각화
    ---------------------
    df: pandas.DataFrame with columns ['pid', 'times', 'varis', 'values', 'mask']
    pid_target: 시각화할 대상 pid
    var_names: dict {var_id: var_name}, 없으면 'var {id}'로 표시
    time_unit: x축 단위 문자열
    figsize_per_plot: 각 subplot 크기 (width, height)
    sharey: True면 y축 스케일을 공유
    """

    # --------- 특정 pid 필터링 ---------
    df_pid = df[df['pid'] == pid_target].copy()
    if df_pid.empty:
        raise ValueError(f"pid={pid_target} 에 해당하는 데이터가 없습니다.")

    # --------- 정렬 (time, varis 오름차순) ---------
    df_pid = df_pid.sort_values(by=['varis'], ascending=[True])

    uniq_vars = df_pid['varis'].unique()
    nplots = len(uniq_vars)
    cols = int(ceil(sqrt(nplots)))
    rows = int(ceil(nplots / cols))

    fig, axes = plt.subplots(rows, cols,
                             figsize=(figsize_per_plot[0]*cols, figsize_per_plot[1]*rows),
                             sharey=sharey)
    if not isinstance(axes, np.ndarray):
        axes = np.array([axes])
    axes = axes.reshape(rows, cols)

    handles, labels = [], []

    for idx, var_id in enumerate(uniq_vars):
        r, c = divmod(idx, cols)
        ax = axes[r, c]

        df_var = df_pid[df_pid['varis'] == var_id]

        # 마스크 상태별로 다른 마커로 시각화
        df_unmasked = df_var[~df_var['mask']]
        df_masked = df_var[df_var['mask']]

        h = None
        if not df_unmasked.empty:
            h = ax.scatter(df_unmasked['times'], df_unmasked['values'],
                           marker='o', label='unmasked', alpha=0.9)
        if not df_masked.empty:
            h = ax.scatter(df_masked['times'], df_masked['values'],
                           marker='x', label='masked', alpha=0.9)

        # 제목 및 축 라벨
        name = var_names.get(var_id, f"var {var_id}") if var_names else f"var {var_id}"
        ax.set_title(name)
        ax.set_xlabel(f"time ({time_unit})")
        ax.set_ylabel("value")
        ax.grid(True, linestyle='--', linewidth=0.5, alpha=0.5)

        # 공통 legend용 handle/label 추출
        for h, l in zip(*ax.get_legend_handles_labels()):
            if l not in labels:
                handles.append(h)
                labels.append(l)

    # 남는 subplot 비우기
    for j in range(nplots, rows*cols):
        r, c = divmod(j, cols)
        axes[r, c].axis('off')

    # --------- 전체 legend 추가 ---------
    fig.legend(handles, labels, loc='lower right', ncol=len(labels),
               frameon=False, fontsize=10)

    fig.suptitle(f"pid={pid_target} per-variable time-series ({graph_name})", y=0.95, fontsize=12)
    fig.tight_layout(rect=[0, 0, 1, 0.95])  # legend와 title 공간 확보
    plt.show()

def compute_mask_ratio_by_var(df, var_names=None):
    """
    변수별(mask=True 기준) 마스킹 비율 요약 +
    각 변수의 unmasked 관측치가 'pid당 평균 몇 개' 존재하는지 추가.

    df columns: ['pid','times','varis','values','mask']
    """

    required = {'pid','varis','mask'}
    if not required.issubset(df.columns):
        raise ValueError(f"df must contain columns {required}")

    # 전체 pid 수
    pid_count = df['pid'].nunique()
    if pid_count == 0:
        raise ValueError("No pid found")

    # ------------------------------
    # 변수별 집계
    # ------------------------------
    g = df.groupby('varis')['mask'].agg(
        count_total='count',
        count_masked='sum'
    ).reset_index()

    g['count_unmasked'] = g['count_total'] - g['count_masked']
    g['mask_ratio'] = g['count_masked'] / g['count_total']

    # ★ pid 당 평균 unmasked 개수
    g['unmasked_per_pid'] = g['count_unmasked'] / pid_count

    # 변수 이름 매핑
    if var_names:
        g['var_name'] = g['varis'].map(lambda x: var_names.get(x, f"var {x}"))
    else:
        g['var_name'] = g['varis'].apply(lambda x: f"var {x}")

    # 컬럼 정리
    g = g[['varis','var_name',
           'count_total','count_masked','count_unmasked',
           'mask_ratio','unmasked_per_pid']]

    g = g.sort_values(by='varis').reset_index(drop=True)

    return g

inv_var_mappings = {
    0: 'gcs',
    1: 'heart_rate',
    2: 'map',
    3: 'resp_rate',
    4: 'temperature',
    5: 'weight',
    6: 'albumin',
    7: 'bilirubin',
    8: 'creatinine',
    9: 'fio2',
    10: 'glucose',
    11: 'hematocrit',
    12: 'lactate',
    13: 'pao2',
    14: 'ph',
    15: 'platelets',
    16: 'sodium',
    17: 'urine',
    18: 'wbc',
    19: 'a10',
    20: 'a_drug',
    21: 'a_supplements',
    22: 'b',
    23: 'c01',
    24: 'c01_etc',
    25: 'c_else',
    26: 'h',
    27: 'l',
    28: 'm',
    29: 'n',
    30: 'r',
    31: 'v',
    32: 'antibiotics',
    33: 'fluid',
    34: 'ventilator',
}


In [ ]:
mask1 = pd.read_feather("./rd_results/surprisevttg_mimic_multi_t80_mask_eicu.feather").sort_values(by=['pid', 'varis'])
mask2 = pd.read_feather("./rd_results/surprisevt_mimic_multi_t140_mask_eicu.feather").sort_values(by=['pid', 'varis'])
mask3 = pd.read_feather("./rd_results/surprise_mimic_multi_t90_mask_eicu.feather").sort_values(by=['pid', 'varis'])
mask4 = pd.read_feather("./rd_results/surprisevttg_mimic_augp2_multi_t80_mask_eicu.feather").sort_values(by=['pid', 'varis'])
mask5 = pd.read_feather("./rd_results/surprisevt_mimic_augp2_multi_t140_mask_eicu.feather").sort_values(by=['pid', 'varis'])
mask6 = pd.read_feather("./rd_results/surprise_mimic_augp2_multi_t90_mask_eicu.feather").sort_values(by=['pid', 'varis'])

# plot_patient_by_var_df(mask1, pid_target=268487,graph_name='VT TG 80', var_names=inv_var_mappings)
# ratio1=compute_mask_ratio_by_var(mask1, var_names=inv_var_mappings)
# plot_patient_by_var_df(mask2, pid_target=268487,graph_name='VT', var_names=inv_var_mappings) 
# ratio2=compute_mask_ratio_by_var(mask2, var_names=inv_var_mappings)
# plot_patient_by_var_df(mask3, pid_target=268487,graph_name='Surprise', var_names=inv_var_mappings) 
# ratio3=compute_mask_ratio_by_var(mask3, var_names=inv_var_mappings)
# plot_patient_by_var_df(mask4, pid_target=268487,graph_name='VT TG 80 (Aug)', var_names=inv_var_mappings)
# ratio4=compute_mask_ratio_by_var(mask4, var_names=inv_var_mappings)
# plot_patient_by_var_df(mask5, pid_target=268487,graph_name='VT (Aug)', var_names=inv_var_mappings) 
# ratio5=compute_mask_ratio_by_var(mask5, var_names=inv_var_mappings)
# plot_patient_by_var_df(mask6, pid_target=268487,graph_name='Surprise (Aug)', var_names=inv_var_mappings) 
# ratio6=compute_mask_ratio_by_var(mask6, var_names=inv_var_mappings)

In [ ]:
var_mse1, overall1, detail1 = masked_reconstruction_mse(
    mask1,
    var_names=inv_var_mappings,
    pid_filter=None,
    pid_target=268487,          # 보고 싶은 pid
)

var_mse2, overall2, detail2 = masked_reconstruction_mse(
    mask2,
    var_names=inv_var_mappings,
    pid_filter=None,
    pid_target=268487,          # 보고 싶은 pid
)

var_mse3, overall3, detail3 = masked_reconstruction_mse(
    mask3,
    var_names=inv_var_mappings,
    pid_filter=None,
    pid_target=268487,          # 보고 싶은 pid
)

var_mse4, overall4, detail4 = masked_reconstruction_mse(
    mask4,
    var_names=inv_var_mappings,
    pid_filter=None,
    pid_target=268487,          # 보고 싶은 pid
)

var_mse5, overall5, detail5 = masked_reconstruction_mse(
    mask5,
    var_names=inv_var_mappings,
    pid_filter=None,
    pid_target=268487,          # 보고 싶은 pid
)

var_mse6, overall6, detail6 = masked_reconstruction_mse(
    mask6,
    var_names=inv_var_mappings,
    pid_filter=None,
    pid_target=268487,          # 보고 싶은 pid
)

In [ ]:
# Epoch checkpoints, eICU
mask1 = pd.read_feather("./rd_results/surprisevttg_mimic_multi_t80_epoch15_mask_eicu.feather").sort_values(by=['pid', 'varis'])
mask2 = pd.read_feather("./rd_results/surprisevttg_mimic_multi_t80_epoch5_mask_eicu.feather").sort_values(by=['pid', 'varis'])
mask3 = pd.read_feather("./rd_results/surprisevttg_mimic_multi_t80_epoch1_mask_eicu.feather").sort_values(by=['pid', 'varis'])
mask4 = pd.read_feather("./rd_results/surprise_mimic_multi_t90_epoch15_mask_eicu.feather").sort_values(by=['pid', 'varis'])
mask5 = pd.read_feather("./rd_results/surprise_mimic_multi_t90_epoch5_mask_eicu.feather").sort_values(by=['pid', 'varis'])
mask6 = pd.read_feather("./rd_results/surprise_mimic_multi_t90_epoch1_mask_eicu.feather").sort_values(by=['pid', 'varis'])

var_mse1, overall1, detail1 = masked_reconstruction_mse(
    mask1,
    var_names=inv_var_mappings,
    pid_filter=None,
    pid_target=268487,          # 보고 싶은 pid
)

var_mse2, overall2, detail2 = masked_reconstruction_mse(
    mask2,
    var_names=inv_var_mappings,
    pid_filter=None,
    pid_target=268487,          # 보고 싶은 pid
)

var_mse3, overall3, detail3 = masked_reconstruction_mse(
    mask3,
    var_names=inv_var_mappings,
    pid_filter=None,
    pid_target=268487,          # 보고 싶은 pid
)

var_mse4, overall4, detail4 = masked_reconstruction_mse(
    mask4,
    var_names=inv_var_mappings,
    pid_filter=None,
    pid_target=268487,          # 보고 싶은 pid
)

var_mse5, overall5, detail5 = masked_reconstruction_mse(
    mask5,
    var_names=inv_var_mappings,
    pid_filter=None,
    pid_target=268487,          # 보고 싶은 pid
)

var_mse6, overall6, detail6 = masked_reconstruction_mse(
    mask6,
    var_names=inv_var_mappings,
    pid_filter=None,
    pid_target=268487,          # 보고 싶은 pid
)

In [ ]:
# Epoch checkpoints, mimic
mask1 = pd.read_feather("./rd_results/surprisevttg_mimic_multi_t80_epoch15_mask_mimic.feather").sort_values(by=['pid', 'varis'])
mask2 = pd.read_feather("./rd_results/surprisevttg_mimic_multi_t80_epoch5_mask_mimic.feather").sort_values(by=['pid', 'varis'])
mask3 = pd.read_feather("./rd_results/surprisevttg_mimic_multi_t80_epoch1_mask_mimic.feather").sort_values(by=['pid', 'varis'])
mask4 = pd.read_feather("./rd_results/surprise_mimic_multi_t90_epoch15_mask_mimic.feather").sort_values(by=['pid', 'varis'])
mask5 = pd.read_feather("./rd_results/surprise_mimic_multi_t90_epoch5_mask_mimic.feather").sort_values(by=['pid', 'varis'])
mask6 = pd.read_feather("./rd_results/surprise_mimic_multi_t90_epoch1_mask_mimic.feather").sort_values(by=['pid', 'varis'])

var_mse1, overall1, detail1 = masked_reconstruction_mse(
    mask1,
    var_names=inv_var_mappings,
    pid_filter=None,
    pid_target=26373139,          # 보고 싶은 pid
)

var_mse2, overall2, detail2 = masked_reconstruction_mse(
    mask2,
    var_names=inv_var_mappings,
    pid_filter=None,
    pid_target=26373139,          # 보고 싶은 pid
)

var_mse3, overall3, detail3 = masked_reconstruction_mse(
    mask3,
    var_names=inv_var_mappings,
    pid_filter=None,
    pid_target=26373139,          # 보고 싶은 pid
)

var_mse4, overall4, detail4 = masked_reconstruction_mse(
    mask4,
    var_names=inv_var_mappings,
    pid_filter=None,
    pid_target=26373139,          # 보고 싶은 pid
)

var_mse5, overall5, detail5 = masked_reconstruction_mse(
    mask5,
    var_names=inv_var_mappings,
    pid_filter=None,
    pid_target=26373139,          # 보고 싶은 pid
)

var_mse6, overall6, detail6 = masked_reconstruction_mse(
    mask6,
    var_names=inv_var_mappings,
    pid_filter=None,
    pid_target=26373139,          # 보고 싶은 pid
)

In [ ]:
source_mask1 = pd.read_feather("./rd_results/surprisevttg_mimic_multi_t80_mask_mimic.feather").sort_values(by=['pid', 'varis'])
source_mask2 = pd.read_feather("./rd_results/surprisevt_mimic_multi_t140_mask_mimic.feather").sort_values(by=['pid', 'varis'])
source_mask3 = pd.read_feather("./rd_results/surprise_mimic_multi_t90_mask_mimic.feather").sort_values(by=['pid', 'varis'])
source_mask4 = pd.read_feather("./rd_results/surprisevttg_mimic_augp2_multi_t80_mask_mimic.feather").sort_values(by=['pid', 'varis'])
source_mask5 = pd.read_feather("./rd_results/surprisevt_mimic_augp2_multi_t140_mask_mimic.feather").sort_values(by=['pid', 'varis'])
source_mask6 = pd.read_feather("./rd_results/surprise_mimic_augp2_multi_t90_mask_mimic.feather").sort_values(by=['pid', 'varis'])


# plot_patient_by_var_df(source_mask1, pid_target=26373139,graph_name='Source VT TG 80', var_names=inv_var_mappings)
# source_ratio1=compute_mask_ratio_by_var(source_mask1, var_names=inv_var_mappings)
# plot_patient_by_var_df(source_mask2, pid_target=26373139,graph_name='Source VT', var_names=inv_var_mappings) 
# source_ratio2=compute_mask_ratio_by_var(source_mask2, var_names=inv_var_mappings)
# plot_patient_by_var_df(source_mask3, pid_target=26373139,graph_name='Source Surprise', var_names=inv_var_mappings) 
# source_ratio3=compute_mask_ratio_by_var(source_mask3, var_names=inv_var_mappings)
# plot_patient_by_var_df(source_mask4, pid_target=26373139,graph_name='VT TG 80 (Aug)', var_names=inv_var_mappings)
# ratio4=compute_mask_ratio_by_var(source_mask4, var_names=inv_var_mappings)
# plot_patient_by_var_df(source_mask5, pid_target=26373139,graph_name='VT (Aug)', var_names=inv_var_mappings) 
# ratio5=compute_mask_ratio_by_var(source_mask5, var_names=inv_var_mappings)
# plot_patient_by_var_df(source_mask6, pid_target=26373139,graph_name='Surprise (Aug)', var_names=inv_var_mappings) 
# ratio6=compute_mask_ratio_by_var(source_mask6, var_names=inv_var_mappings)


In [ ]:
var_mses1, overalls1, details1 = masked_reconstruction_mse(
    source_mask1,
    var_names=inv_var_mappings,
    pid_filter=None,
    pid_target=26373139,          # 보고 싶은 pid
)

var_mses2, overalls2, details2 = masked_reconstruction_mse(
    source_mask2,
    var_names=inv_var_mappings,
    pid_filter=None,
    pid_target=26373139,          # 보고 싶은 pid
)

var_mses3, overalls3, details3 = masked_reconstruction_mse(
    source_mask3,
    var_names=inv_var_mappings,
    pid_filter=None,
    pid_target=26373139,          # 보고 싶은 pid
)

var_mse4, overall4, detail4 = masked_reconstruction_mse(
    source_mask4,
    var_names=inv_var_mappings,
    pid_filter=None,
    pid_target=26373139,          # 보고 싶은 pid
)

var_mse5, overall5, detail5 = masked_reconstruction_mse(
    source_mask5,
    var_names=inv_var_mappings,
    pid_filter=None,
    pid_target=26373139,          # 보고 싶은 pid
)

var_mse6, overall6, detail6 = masked_reconstruction_mse(
    source_mask6,
    var_names=inv_var_mappings,
    pid_filter=None,
    pid_target=26373139,          # 보고 싶은 pid
)

In [ ]:
source_scores_1 = pd.read_feather("./rd_results/surprise_mimic_multi_t90_mimic_test_pid26373139_scores.feather")#.sort_values(by=['pid', 'varis'])
target_scores_1 = pd.read_feather("./rd_results/surprise_mimic_multi_t90_eicu_target_pid268487_scores.feather")#.sort_values(by=['pid', 'varis'])
source_scores_2 = pd.read_feather("./rd_results/surprisevt_mimic_multi_t140_mimic_test_pid26373139_scores.feather")#.sort_values(by=['pid', 'varis'])
target_scores_2 = pd.read_feather("./rd_results/surprisevt_mimic_multi_t140_eicu_target_pid268487_scores.feather")#.sort_values(by=['pid', 'varis'])
source_scores_3 = pd.read_feather("./rd_results/surprisevttg_mimic_multi_t80_mimic_test_pid26373139_scores.feather")#.sort_values(by=['pid', 'varis'])
target_scores_3 = pd.read_feather("./rd_results/surprisevttg_mimic_multi_t80_eicu_target_pid268487_scores.feather")#.sort_values(by=['pid', 'varis'])


In [ ]:
test1 = compare_unmasked_counts_unpaired(source_mask1, mask1).sort_values(by='varis')
test1

In [ ]:
test2 = compare_unmasked_counts_unpaired(source_mask2, mask2).sort_values(by='varis')
test2

In [ ]:
test3 = compare_unmasked_counts_unpaired(source_mask3, mask3).sort_values(by='varis')
test3

In [ ]:
overall = pd.DataFrame([overall1, overall2, overall3])
overalls = pd.DataFrame([overalls1, overalls2, overalls3])

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score
from scipy.special import expit  # sigmoid
import re

emb_df = pd.read_feather("./rd_results/strats_mimic_multi_embeddings_target.feather")

data = pd.read_feather("./data/eicu_data_test.feather")
summary = (
    data.groupby("pid")
        .agg(
            n_obs=("offset", "size"),
            n_unique_times=("offset", "nunique"),
        )
        .reset_index()
)
def infer_label_cols(df):
    cols = df.columns

    # exclude: pid, source
    exclude = {"pid", "source"}

    # exclude embeddings e0~e63
    embed_cols = {c for c in cols if re.fullmatch(r"e\d+", str(c))}

    # exclude predictions pred_*
    pred_cols = {c for c in cols if str(c).startswith("pred_")}

    label_cols = [c for c in cols if c not in exclude and c not in embed_cols and c not in pred_cols]
    return label_cols

label_cols = infer_label_cols(emb_df)
print(len(label_cols), label_cols[:5])


def compute_multilabel_metrics_safe(df, label_cols, threshold=0.5):
    if df.empty:
        return {
            "AUROC_macro": np.nan,
            "AUPRC_macro": np.nan,
            "ACC_macro":   np.nan,
            "n_samples":   0,
            "n_labels_used": 0,
            "n_labels_skipped": 0,
            "skipped_labels": [],
        }

    y_true = df[label_cols].to_numpy()
    y_pred = df[[f"pred_{c}" for c in label_cols]].to_numpy()  # logits or scores

    # ensure 2D
    if y_true.ndim == 1:
        y_true = y_true[:, None]
    if y_pred.ndim == 1:
        y_pred = y_pred[:, None]

    K = y_true.shape[1]
    aurocs, auprcs, accs = [], [], []
    skipped = []

    y_prob = expit(y_pred)
    y_hat = (y_prob >= threshold).astype(int)

    for k in range(K):
        yt = y_true[:, k]
        yp = y_pred[:, k]   # scores for AUROC/AUPRC
        yh = y_hat[:, k]

        if np.unique(yt).size < 2:
            skipped.append(label_cols[k])
            continue

        aurocs.append(roc_auc_score(yt, yp))
        auprcs.append(average_precision_score(yt, yp))
        accs.append(accuracy_score(yt, yh))

    return {
        "AUROC_macro": float(np.mean(aurocs)) if aurocs else np.nan,
        "AUPRC_macro": float(np.mean(auprcs)) if auprcs else np.nan,
        "ACC_macro":   float(np.mean(accs))   if accs else np.nan,
        "n_samples": int(len(df)),
        "n_labels_used": int(len(aurocs)),
        "n_labels_skipped": int(len(skipped)),
        "skipped_labels": skipped[:10],
    }


label_cols = infer_label_cols(emb_df)

# ----- 구간 정의 -----
bins = {
    "all": summary["pid"].values,  # 전체
    "n_obs < 800": summary.loc[summary["n_obs"] < 800, "pid"].values,
    "800 <= n_obs < 1200": summary.loc[(summary["n_obs"] >= 800) & (summary["n_obs"] < 1200), "pid"].values,
    "1200 <= n_obs < 1400": summary.loc[(summary["n_obs"] >= 1200) & (summary["n_obs"] < 1400), "pid"].values,
    "1400 <= n_obs < 1800": summary.loc[(summary["n_obs"] >= 1400) & (summary["n_obs"] < 1800), "pid"].values,
    "n_obs >= 1800": summary.loc[summary["n_obs"] >= 1800, "pid"].values,
}

metrics_dict = {}

for name, pid_list in bins.items():
    if name == "all":
        df_group = emb_df
    else:
        df_group = emb_df[emb_df["pid"].isin(pid_list)]

    m = compute_multilabel_metrics_safe(df_group, label_cols)
    metrics_dict[name] = m

metrics_all = pd.DataFrame(metrics_dict).T
metrics_all

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score
from scipy.special import expit  # sigmoid
import re

emb_df = pd.read_feather("./rd_results/surprise_mimic_multi_t90_embeddings_target.feather")

data = pd.read_feather("./data/eicu_data_test.feather")
summary = (
    data.groupby("pid")
        .agg(
            n_obs=("offset", "size"),
            n_unique_times=("offset", "nunique"),
        )
        .reset_index()
)
def infer_label_cols(df):
    cols = df.columns

    # exclude: pid, source
    exclude = {"pid", "source"}

    # exclude embeddings e0~e63
    embed_cols = {c for c in cols if re.fullmatch(r"e\d+", str(c))}

    # exclude predictions pred_*
    pred_cols = {c for c in cols if str(c).startswith("pred_")}

    label_cols = [c for c in cols if c not in exclude and c not in embed_cols and c not in pred_cols]
    return label_cols

label_cols = infer_label_cols(emb_df)
print(len(label_cols), label_cols[:5])


def compute_multilabel_metrics_safe(df, label_cols, threshold=0.5):
    if df.empty:
        return {
            "AUROC_macro": np.nan,
            "AUPRC_macro": np.nan,
            "ACC_macro":   np.nan,
            "n_samples":   0,
            "n_labels_used": 0,
            "n_labels_skipped": 0,
            "skipped_labels": [],
        }

    y_true = df[label_cols].to_numpy()
    y_pred = df[[f"pred_{c}" for c in label_cols]].to_numpy()  # logits or scores

    # ensure 2D
    if y_true.ndim == 1:
        y_true = y_true[:, None]
    if y_pred.ndim == 1:
        y_pred = y_pred[:, None]

    K = y_true.shape[1]
    aurocs, auprcs, accs = [], [], []
    skipped = []

    y_prob = expit(y_pred)
    y_hat = (y_prob >= threshold).astype(int)

    for k in range(K):
        yt = y_true[:, k]
        yp = y_pred[:, k]   # scores for AUROC/AUPRC
        yh = y_hat[:, k]

        if np.unique(yt).size < 2:
            skipped.append(label_cols[k])
            continue

        aurocs.append(roc_auc_score(yt, yp))
        auprcs.append(average_precision_score(yt, yp))
        accs.append(accuracy_score(yt, yh))

    return {
        "AUROC_macro": float(np.mean(aurocs)) if aurocs else np.nan,
        "AUPRC_macro": float(np.mean(auprcs)) if auprcs else np.nan,
        "ACC_macro":   float(np.mean(accs))   if accs else np.nan,
        "n_samples": int(len(df)),
        "n_labels_used": int(len(aurocs)),
        "n_labels_skipped": int(len(skipped)),
        "skipped_labels": skipped[:10],
    }


label_cols = infer_label_cols(emb_df)

# ----- 구간 정의 -----
bins = {
    "all": summary["pid"].values,  # 전체
    "n_obs < 800": summary.loc[summary["n_obs"] < 800, "pid"].values,
    "800 <= n_obs < 1200": summary.loc[(summary["n_obs"] >= 800) & (summary["n_obs"] < 1200), "pid"].values,
    "1200 <= n_obs < 1400": summary.loc[(summary["n_obs"] >= 1200) & (summary["n_obs"] < 1400), "pid"].values,
    "1400 <= n_obs < 1800": summary.loc[(summary["n_obs"] >= 1400) & (summary["n_obs"] < 1800), "pid"].values,
    "n_obs >= 1800": summary.loc[summary["n_obs"] >= 1800, "pid"].values,
}

metrics_dict = {}

for name, pid_list in bins.items():
    if name == "all":
        df_group = emb_df
    else:
        df_group = emb_df[emb_df["pid"].isin(pid_list)]

    m = compute_multilabel_metrics_safe(df_group, label_cols)
    metrics_dict[name] = m

metrics_all = pd.DataFrame(metrics_dict).T
metrics_all

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score
from scipy.special import expit  # sigmoid
import re

emb_df = pd.read_feather("./rd_results/surprisevt_mimic_multi_t140_embeddings_target.feather")

data = pd.read_feather("./data/eicu_data_test.feather")
summary = (
    data.groupby("pid")
        .agg(
            n_obs=("offset", "size"),
            n_unique_times=("offset", "nunique"),
        )
        .reset_index()
)
def infer_label_cols(df):
    cols = df.columns

    # exclude: pid, source
    exclude = {"pid", "source"}

    # exclude embeddings e0~e63
    embed_cols = {c for c in cols if re.fullmatch(r"e\d+", str(c))}

    # exclude predictions pred_*
    pred_cols = {c for c in cols if str(c).startswith("pred_")}

    label_cols = [c for c in cols if c not in exclude and c not in embed_cols and c not in pred_cols]
    return label_cols

label_cols = infer_label_cols(emb_df)
print(len(label_cols), label_cols[:5])


def compute_multilabel_metrics_safe(df, label_cols, threshold=0.5):
    if df.empty:
        return {
            "AUROC_macro": np.nan,
            "AUPRC_macro": np.nan,
            "ACC_macro":   np.nan,
            "n_samples":   0,
            "n_labels_used": 0,
            "n_labels_skipped": 0,
            "skipped_labels": [],
        }

    y_true = df[label_cols].to_numpy()
    y_pred = df[[f"pred_{c}" for c in label_cols]].to_numpy()  # logits or scores

    # ensure 2D
    if y_true.ndim == 1:
        y_true = y_true[:, None]
    if y_pred.ndim == 1:
        y_pred = y_pred[:, None]

    K = y_true.shape[1]
    aurocs, auprcs, accs = [], [], []
    skipped = []

    y_prob = expit(y_pred)
    y_hat = (y_prob >= threshold).astype(int)

    for k in range(K):
        yt = y_true[:, k]
        yp = y_pred[:, k]   # scores for AUROC/AUPRC
        yh = y_hat[:, k]

        if np.unique(yt).size < 2:
            skipped.append(label_cols[k])
            continue

        aurocs.append(roc_auc_score(yt, yp))
        auprcs.append(average_precision_score(yt, yp))
        accs.append(accuracy_score(yt, yh))

    return {
        "AUROC_macro": float(np.mean(aurocs)) if aurocs else np.nan,
        "AUPRC_macro": float(np.mean(auprcs)) if auprcs else np.nan,
        "ACC_macro":   float(np.mean(accs))   if accs else np.nan,
        "n_samples": int(len(df)),
        "n_labels_used": int(len(aurocs)),
        "n_labels_skipped": int(len(skipped)),
        "skipped_labels": skipped[:10],
    }


label_cols = infer_label_cols(emb_df)

# ----- 구간 정의 -----
bins = {
    "all": summary["pid"].values,  # 전체
    "n_obs < 800": summary.loc[summary["n_obs"] < 800, "pid"].values,
    "800 <= n_obs < 1200": summary.loc[(summary["n_obs"] >= 800) & (summary["n_obs"] < 1200), "pid"].values,
    "1200 <= n_obs < 1400": summary.loc[(summary["n_obs"] >= 1200) & (summary["n_obs"] < 1400), "pid"].values,
    "1400 <= n_obs < 1800": summary.loc[(summary["n_obs"] >= 1400) & (summary["n_obs"] < 1800), "pid"].values,
    "n_obs >= 1800": summary.loc[summary["n_obs"] >= 1800, "pid"].values,
}

metrics_dict = {}

for name, pid_list in bins.items():
    if name == "all":
        df_group = emb_df
    else:
        df_group = emb_df[emb_df["pid"].isin(pid_list)]

    m = compute_multilabel_metrics_safe(df_group, label_cols)
    metrics_dict[name] = m

metrics_all = pd.DataFrame(metrics_dict).T
metrics_all

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score
from scipy.special import expit  # sigmoid
import re

emb_df = pd.read_feather("./rd_results/surprisevttg_mimic_multi_t80_embeddings_target.feather")

data = pd.read_feather("./data/eicu_data_test.feather")
summary = (
    data.groupby("pid")
        .agg(
            n_obs=("offset", "size"),
            n_unique_times=("offset", "nunique"),
        )
        .reset_index()
)
def infer_label_cols(df):
    cols = df.columns

    # exclude: pid, source
    exclude = {"pid", "source"}

    # exclude embeddings e0~e63
    embed_cols = {c for c in cols if re.fullmatch(r"e\d+", str(c))}

    # exclude predictions pred_*
    pred_cols = {c for c in cols if str(c).startswith("pred_")}

    label_cols = [c for c in cols if c not in exclude and c not in embed_cols and c not in pred_cols]
    return label_cols

label_cols = infer_label_cols(emb_df)
print(len(label_cols), label_cols[:5])


def compute_multilabel_metrics_safe(df, label_cols, threshold=0.5):
    if df.empty:
        return {
            "AUROC_macro": np.nan,
            "AUPRC_macro": np.nan,
            "ACC_macro":   np.nan,
            "n_samples":   0,
            "n_labels_used": 0,
            "n_labels_skipped": 0,
            "skipped_labels": [],
        }

    y_true = df[label_cols].to_numpy()
    y_pred = df[[f"pred_{c}" for c in label_cols]].to_numpy()  # logits or scores

    # ensure 2D
    if y_true.ndim == 1:
        y_true = y_true[:, None]
    if y_pred.ndim == 1:
        y_pred = y_pred[:, None]

    K = y_true.shape[1]
    aurocs, auprcs, accs = [], [], []
    skipped = []

    y_prob = expit(y_pred)
    y_hat = (y_prob >= threshold).astype(int)

    for k in range(K):
        yt = y_true[:, k]
        yp = y_pred[:, k]   # scores for AUROC/AUPRC
        yh = y_hat[:, k]

        if np.unique(yt).size < 2:
            skipped.append(label_cols[k])
            continue

        aurocs.append(roc_auc_score(yt, yp))
        auprcs.append(average_precision_score(yt, yp))
        accs.append(accuracy_score(yt, yh))

    return {
        "AUROC_macro": float(np.mean(aurocs)) if aurocs else np.nan,
        "AUPRC_macro": float(np.mean(auprcs)) if auprcs else np.nan,
        "ACC_macro":   float(np.mean(accs))   if accs else np.nan,
        "n_samples": int(len(df)),
        "n_labels_used": int(len(aurocs)),
        "n_labels_skipped": int(len(skipped)),
        "skipped_labels": skipped[:10],
    }


label_cols = infer_label_cols(emb_df)

# ----- 구간 정의 -----
bins = {
    "all": summary["pid"].values,  # 전체
    "n_obs < 800": summary.loc[summary["n_obs"] < 800, "pid"].values,
    "800 <= n_obs < 1200": summary.loc[(summary["n_obs"] >= 800) & (summary["n_obs"] < 1200), "pid"].values,
    "1200 <= n_obs < 1400": summary.loc[(summary["n_obs"] >= 1200) & (summary["n_obs"] < 1400), "pid"].values,
    "1400 <= n_obs < 1800": summary.loc[(summary["n_obs"] >= 1400) & (summary["n_obs"] < 1800), "pid"].values,
    "n_obs >= 1800": summary.loc[summary["n_obs"] >= 1800, "pid"].values,
}

metrics_dict = {}

for name, pid_list in bins.items():
    if name == "all":
        df_group = emb_df
    else:
        df_group = emb_df[emb_df["pid"].isin(pid_list)]

    m = compute_multilabel_metrics_safe(df_group, label_cols)
    metrics_dict[name] = m

metrics_all = pd.DataFrame(metrics_dict).T
metrics_all

In [ ]:
import pacmap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ----- 임베딩 컬럼 -----
emb_cols = [f"e{i}" for i in range(64)]

df_a = pd.read_feather("./rd_results/strats_mimic_multi_embeddings_test.feather")
df_b = pd.read_feather("./rd_results/strats_mimic_multi_embeddings_target.feather")


# ----- 임베딩 추출 -----
X_a = df_a[emb_cols].values
X_b = df_b[emb_cols].values

# ----- 병합 (같은 공간에 투영하기 위함) -----
X = np.vstack([X_a, X_b])
labels = np.array(
    ["MIMIC"] * len(X_a) +
    ["eICU"] * len(X_b)
)

# ----- PaCMAP -----
reducer = pacmap.PaCMAP(
    n_components=2,
    n_neighbors=15,
    MN_ratio=0.5,
    FP_ratio=2.0,
    random_state=42,
)

X_2d = reducer.fit_transform(X)

# ----- 다시 분리 -----
Xa_2d = X_2d[:len(X_a)]
Xb_2d = X_2d[len(X_a):]

# ----- Plot -----
plt.figure(figsize=(7, 6))
plt.scatter(
    Xa_2d[:, 0], Xa_2d[:, 1],
    s=8, alpha=0.5, color="#2D2926",label="MIMIC"
)
plt.scatter(
    Xb_2d[:, 0], Xb_2d[:, 1],
    s=8, alpha=0.5, color="#ed6f63", label="eICU"
)


plt.legend()
plt.title("PaCMAP of STraTS embeddings (e0–e63)")
plt.xlabel("PaCMAP-1")
plt.ylabel("PaCMAP-2")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import pacmap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ----- 임베딩 컬럼 -----
emb_cols = [f"e{i}" for i in range(64)]

df_a = pd.read_feather("./rd_results/surprise_mimic_multi_t90_embeddings_test.feather")
df_b = pd.read_feather("./rd_results/surprise_mimic_multi_t90_embeddings_target.feather")


# ----- 임베딩 추출 -----
X_a = df_a[emb_cols].values
X_b = df_b[emb_cols].values

# ----- 병합 (같은 공간에 투영하기 위함) -----
X = np.vstack([X_a, X_b])
labels = np.array(
    ["MIMIC"] * len(X_a) +
    ["eICU"] * len(X_b)
)

# ----- PaCMAP -----
reducer = pacmap.PaCMAP(
    n_components=2,
    n_neighbors=15,
    MN_ratio=0.5,
    FP_ratio=2.0,
    random_state=42,
)

X_2d = reducer.fit_transform(X)

# ----- 다시 분리 -----
Xa_2d = X_2d[:len(X_a)]
Xb_2d = X_2d[len(X_a):]

# ----- Plot -----
plt.figure(figsize=(7, 6))
plt.scatter(
    Xa_2d[:, 0], Xa_2d[:, 1],
    s=8, alpha=0.5, color="#2D2926",label="MIMIC"
)
plt.scatter(
    Xb_2d[:, 0], Xb_2d[:, 1],
    s=8, alpha=0.5, color="#ed6f63", label="eICU"
)

plt.legend()
plt.title("PaCMAP of Surprise embeddings (e0–e63)")
plt.xlabel("PaCMAP-1")
plt.ylabel("PaCMAP-2")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import pacmap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ----- 임베딩 컬럼 -----
emb_cols = [f"e{i}" for i in range(64)]

df_a = pd.read_feather("./rd_results/surprisevt_mimic_multi_t140_embeddings_test.feather")
df_b = pd.read_feather("./rd_results/surprisevt_mimic_multi_t140_embeddings_target.feather")


# ----- 임베딩 추출 -----
X_a = df_a[emb_cols].values
X_b = df_b[emb_cols].values

# ----- 병합 (같은 공간에 투영하기 위함) -----
X = np.vstack([X_a, X_b])
labels = np.array(
    ["MIMIC"] * len(X_a) +
    ["eICU"] * len(X_b)
)

# ----- PaCMAP -----
reducer = pacmap.PaCMAP(
    n_components=2,
    n_neighbors=15,
    MN_ratio=0.5,
    FP_ratio=2.0,
    random_state=42,
)

X_2d = reducer.fit_transform(X)

# ----- 다시 분리 -----
Xa_2d = X_2d[:len(X_a)]
Xb_2d = X_2d[len(X_a):]

# ----- Plot -----
plt.figure(figsize=(7, 6))
plt.scatter(
    Xa_2d[:, 0], Xa_2d[:, 1],
    s=8, alpha=0.5, color="#2D2926",label="MIMIC"
)
plt.scatter(
    Xb_2d[:, 0], Xb_2d[:, 1],
    s=8, alpha=0.5, color="#ed6f63", label="eICU"
)

plt.legend()
plt.title("PaCMAP of VT embeddings (e0–e63)")
plt.xlabel("PaCMAP-1")
plt.ylabel("PaCMAP-2")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import pacmap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ----- 임베딩 컬럼 -----
emb_cols = [f"e{i}" for i in range(64)]

df_a = pd.read_feather("./rd_results/surprisevttg_mimic_multi_t80_embeddings_test.feather")
df_b = pd.read_feather("./rd_results/surprisevttg_mimic_multi_t80_embeddings_target.feather")


# ----- 임베딩 추출 -----
X_a = df_a[emb_cols].values
X_b = df_b[emb_cols].values

# ----- 병합 (같은 공간에 투영하기 위함) -----
X = np.vstack([X_a, X_b])
labels = np.array(
    ["MIMIC"] * len(X_a) +
    ["eICU"] * len(X_b)
)

# ----- PaCMAP -----
reducer = pacmap.PaCMAP(
    n_components=2,
    n_neighbors=15,
    MN_ratio=0.5,
    FP_ratio=2.0,
    random_state=42,
)

X_2d = reducer.fit_transform(X)

# ----- 다시 분리 -----
Xa_2d = X_2d[:len(X_a)]
Xb_2d = X_2d[len(X_a):]

# ----- Plot -----
plt.figure(figsize=(7, 6))
plt.scatter(
    Xa_2d[:, 0], Xa_2d[:, 1],
    s=8, alpha=0.5, color="#2D2926",label="MIMIC"
)
plt.scatter(
    Xb_2d[:, 0], Xb_2d[:, 1],
    s=8, alpha=0.5, color="#ed6f63", label="eICU"
)

plt.legend()
plt.title("PaCMAP of VTTG embeddings (e0–e63)")
plt.xlabel("PaCMAP-1")
plt.ylabel("PaCMAP-2")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
data_m = pd.read_feather("./data/mimic_data_valid.feather")
data_e = pd.read_feather("./data/eicu_data_valid.feather")
data_a = pd.read_feather("./data/mimic_data_aug_valid.feather")
data_ap = pd.read_feather("./data/mimic_data_aug_p_valid.feather")

def summarize_item_distribution(df, inv_var_mapping, dataset_name="dataset"):
    # PID, itemid별 observation 수 세기
    cnt = df.groupby(["pid", "itemid"]).size().reset_index(name="n_obs")

    # PID별로 itemid 개수 (unique 변수 수)
    var_per_pid = cnt.groupby("pid")["itemid"].nunique()

    # itemid별로 PID에 대해 관측치 개수의 평균/표준편차
    stats = cnt.groupby("itemid")["n_obs"].agg(
        mean_obs_per_pid="mean",
        std_obs_per_pid="std"
    ).reset_index()

    # 변수 이름 매핑
    stats["var_name"] = stats["itemid"].map(inv_var_mapping).fillna("unknown_var")

    # PID별 itemid 개수 전체 요약도 함께 출력
    print(f"\n[{dataset_name}] PID별 unique itemid 분포:")
    print(f"- Mean: {var_per_pid.mean():.2f}")
    print(f"- Std : {var_per_pid.std():.2f}")
    print(f"- Min : {var_per_pid.min():.0f}")
    print(f"- Max : {var_per_pid.max():.0f}")
    print(f"- Count: {len(var_per_pid)} PIDs")

    return stats.sort_values("mean_obs_per_pid", ascending=False)

# 실행
stats_m = summarize_item_distribution(data_m, inv_var_mappings, dataset_name="MIMIC")
stats_e = summarize_item_distribution(data_e, inv_var_mappings, dataset_name="eICU")
stats_a = summarize_item_distribution(data_a, inv_var_mappings, dataset_name="MIMIC Aug")


In [ ]:
s1 = np.load('./processed_data/P19data/processed_data/PT_dict_list_6.npy', allow_pickle=True)

example = s1[21668]
example2 = s1[50]

times = [s1[i]['length'] for i in range(0,len(s1))]

In [ ]:
s2 = np.load('./processed_data/P12data/processed_data/PTdict_list.npy', allow_pickle=True)

example = s2[0]

In [ ]:
oc = np.load('./processed_data/P19data/processed_data/arr_outcomes_6.npy', allow_pickle=True)
oc12 = np.load('./processed_data/P12data/processed_data/arr_outcomes.npy', allow_pickle=True)
